# 04 Homework 04 ETM — Node Classification

**Task 5 of the Final Assignment**

This notebook builds a **CellComplex** from the Brownstone floor plan room volumes, assigns
room-type labels and door-type apertures, exports the graph to CSV in the MSD model schema,
then runs the pretrained `msd_node_classifier.pt` to predict each room's type.

## Pipeline
1. Load room OBJs → build CellComplex with room-type labels  
2. Load door OBJs → add as apertures  
3. `Graph.ByTopology(cc, directApertures=True)` → adjacency graph through doors  
4. Compute zoning and connectivity one-hot features per node/edge  
5. Export to CSV (MSD schema)  
6. Load with `PyG.ByCSVPath`, load pretrained model, predict  
7. Visualise true vs predicted labels

## MSD label → room-type mapping
| `room_type` | `label` | Zoning class |
|---|---:|---|
| `bedroom` | `0` | Private / static |
| `livingroom` | `1` | Living / dynamic |
| `kitchen` | `2` | Living / dynamic |
| `dining` | `3` | Living / dynamic |
| `corridor` | `4` | Living / dynamic |
| `stairs` | `5` | Service / functional |
| `storeroom` | `6` | Service / functional |
| `bathroom` | `7` | Service / functional |
| `balcony` | `8` | Outdoor / semi-outdoor |

## 1. Imports

In [160]:
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Graph import Graph
from topologicpy.Color import Color
from topologicpy.Helper import Helper
from topologicpy.PyG import PyG

import pandas as pd
import os
from pathlib import Path

## 2. Check TopologicPy version

In [161]:
print("This notebook requires topologicpy version 0.9.43 or newer.")
print(Helper.Version())

This notebook requires topologicpy version 0.9.43 or newer.
The version that you are using (0.9.43) is OLDER than the latest version (0.9.50) from PyPI. Please consider upgrading to the latest version.


## 3. Set renderer
* VS Code: `"vscode"`  
* Google Colab: `"colab"`  
* Browser: `"browser"`

In [162]:
renderer = "vscode"

## 4. Room-type and door-type mappings

### Zoning classes (for node features)
| Zoning | One-hot index | Room types |
|---|---:|---|
| Private/static | `0` | bedroom |
| Living/dynamic | `1` | livingroom, kitchen, dining, corridor |
| Service/functional | `2` | stairs, storeroom, bathroom |
| Outdoor/semi-outdoor | `3` | balcony |

### Connectivity classes (for node and edge features)
| Connection type | One-hot index | Meaning |
|---|---:|---|
| passage | `0` | Open opening / corridor connection |
| door | `1` | Standard interior door |
| entrance_door | `2` | Exterior / entrance door |

In [163]:
# --- Room type → integer label ---
ROOM_LABEL = {
    "bedroom":     0,
    "livingroom":  1,
    "kitchen":     2,
    "dining":      3,
    "corridor":    4,
    "stairs":      5,
    "storeroom":   6,
    "bathroom":    7,
    "balcony":     8,
}

# --- Room type → zoning one-hot [private, living, service, outdoor] ---
ZONING = {
    "bedroom":     [1, 0, 0, 0],
    "livingroom":  [0, 1, 0, 0],
    "kitchen":     [0, 1, 0, 0],
    "dining":      [0, 1, 0, 0],
    "corridor":    [0, 1, 0, 0],
    "stairs":      [0, 0, 1, 0],
    "storeroom":   [0, 0, 1, 0],
    "bathroom":    [0, 0, 1, 0],
    "balcony":     [0, 0, 0, 1],
}

# --- Room type → node connectivity one-hot [passage, door, entrance_door] ---
NODE_CONNECTIVITY = {
    "bedroom":     [0, 1, 0],
    "livingroom":  [0, 1, 0],
    "kitchen":     [0, 1, 0],
    "dining":      [0, 1, 0],
    "corridor":    [1, 0, 0],
    "stairs":      [1, 0, 0],
    "storeroom":   [0, 1, 0],
    "bathroom":    [0, 1, 0],
    "balcony":     [0, 0, 0],
}

# --- Door OBJ type → edge connectivity one-hot [passage, door, entrance_door] ---
DOOR_CONNECTIVITY = {
    "passage":       [1, 0, 0],
    "door":          [0, 1, 0],
    "entrance_door": [0, 0, 1],
}

# --- Color for visualization ---
ROOM_COLOR = {
    "bedroom":    "#4E79A7",
    "livingroom": "#F28E2B",
    "kitchen":    "#E15759",
    "dining":     "#76B7B2",
    "corridor":   "#59A14F",
    "stairs":     "#EDC948",
    "storeroom":  "#B07AA1",
    "bathroom":   "#FF9DA7",
    "balcony":    "#9C755F",
    "unknown":    "#AAAAAA",
}

print("Mappings loaded.")
print(f"  {len(ROOM_LABEL)} room types, {len(DOOR_CONNECTIVITY)} door types")

Mappings loaded.
  9 room types, 3 door types


## 5. Paths

In [164]:
OBJECTS_DIR  = r"C:\Users\etmaglari\IAAC\etmaglari_gML\Homework04\Objects"
MODEL_PATH   = r"C:\Users\etmaglari\IAAC\etmaglari_gML\S0 Classes\msd-main\msd_node_classifier.pt"
DATASET_PATH = r"C:\Users\etmaglari\IAAC\etmaglari_gML\Homework04\dataset_node_classification"
PRED_CSV     = os.path.join(DATASET_PATH, "node_predictions_homework04.csv")

os.makedirs(DATASET_PATH, exist_ok=True)
print(f"Objects dir : {OBJECTS_DIR}")
print(f"Model path  : {MODEL_PATH}")
print(f"Dataset path: {DATASET_PATH}")

Objects dir : C:\Users\etmaglari\IAAC\etmaglari_gML\Homework04\Objects
Model path  : C:\Users\etmaglari\IAAC\etmaglari_gML\S0 Classes\msd-main\msd_node_classifier.pt
Dataset path: C:\Users\etmaglari\IAAC\etmaglari_gML\Homework04\dataset_node_classification


## 6. Load room OBJs and create labelled selectors

Each OBJ file represents one room type. We import the geometry, extract the enclosed cell
volumes, and create **selector vertices** (internal points) carrying `room_type`, `label`,
`cell_color`, `zoning`, and `connectivity` dictionaries. These selectors are later used to
transfer labels onto the merged CellComplex cells.

In [165]:
from collections import defaultdict

# Exact filenames (without .obj) -> MSD room_type string
OBJ_TO_ROOM = {
    "Bedroom":     "bedroom",
    "Living room": "livingroom",
    "Kitchen":     "kitchen",
    "Corridor":    "corridor",
    "Stair":       "stairs",
    "Bathroom":    "bathroom",
}

# ── OBJ parser ────────────────────────────────────────────────────────────────

def _parse_obj(path):
    verts   = []
    objects = []
    cur_name, cur_f, cur_l = "obj", [], []
    try:
        with open(path, "r", encoding="utf-8", errors="ignore") as fh:
            for line in fh:
                ln = line.strip()
                if not ln or ln.startswith("#"):
                    continue
                tok = ln.split()
                if tok[0] == "v" and len(tok) >= 4:
                    ox, oy, oz = float(tok[1]), float(tok[2]), float(tok[3])
                    verts.append((ox, -oz, oy))   # +90 deg around X -> Z-up
                elif tok[0] in ("o", "g"):
                    if cur_f or cur_l:
                        objects.append((cur_name, cur_f, cur_l))
                        cur_f, cur_l = [], []
                    cur_name = tok[1] if len(tok) > 1 else "obj"
                elif tok[0] == "f":
                    idx = [int(p.split("/")[0]) - 1 for p in tok[1:]]
                    cur_f.append(idx)
                elif tok[0] == "l":
                    idx = [int(p) - 1 for p in tok[1:]]
                    cur_l.append(idx)
    except Exception as e:
        print(f"    parse error ({path}): {e}")
    if cur_f or cur_l:
        objects.append((cur_name, cur_f, cur_l))
    return verts, objects

# ── face builder helpers ──────────────────────────────────────────────────────

def _make_face(fv):
    n = len(fv)
    if n < 3:
        return None
    try:
        edges = [Edge.ByVertices([fv[i], fv[(i + 1) % n]]) for i in range(n)]
        if all(edges):
            w = Wire.ByEdges(edges)
            if w:
                return Face.ByWire(w)
    except Exception:
        pass
    return None

def _fan_tris(fv):
    tris = []
    v0 = fv[0]
    for i in range(1, len(fv) - 1):
        f = _make_face([v0, fv[i], fv[i + 1]])
        if f:
            tris.append(f)
    return tris

def _faces_to_cell(face_list):
    if not face_list:
        return None
    try:
        merged = Topology.SelfMerge(Cluster.ByTopologies(face_list))
        mc = Topology.Cells(merged)
        if mc:
            return mc[0]
        if Topology.IsInstance(merged, "Cell"):
            return merged
        if Topology.IsInstance(merged, "Shell"):
            return Cell.ByShell(merged)
    except Exception:
        pass
    return None

# ── display helper ────────────────────────────────────────────────────────────

def _vkey(v, tol=3):
    return tuple(round(x, tol) for x in Vertex.Coordinates(v))

def _normal_key(f, tol=1):
    n = Face.Normal(f)
    return tuple(round(x, tol) for x in n) if n else (0, 0, 1)

def merge_coplanar_for_display(faces):
    groups = defaultdict(list)
    for f in faces:
        groups[_normal_key(f)].append(f)
    clean = []
    for _, grp in groups.items():
        if len(grp) == 1:
            clean.append(grp[0])
            continue
        edge_cnt, edge_obj = defaultdict(int), {}
        for f in grp:
            for e in (Topology.Edges(f) or []):
                vs = Topology.Vertices(e)
                k  = tuple(sorted([_vkey(vs[0]), _vkey(vs[1])]))
                edge_cnt[k] += 1
                edge_obj[k]  = e
        boundary = [edge_obj[k] for k, cnt in edge_cnt.items() if cnt == 1]
        if not boundary:
            clean.extend(grp)
            continue
        try:
            mw    = Topology.SelfMerge(Cluster.ByTopologies(boundary))
            wires = Topology.Wires(mw) or []
            if wires:
                wires.sort(key=lambda w: len(Topology.Vertices(w) or []), reverse=True)
                f = Face.ByWire(wires[0])
                if f:
                    clean.append(f)
                    continue
        except Exception:
            pass
        clean.extend(grp)
    return clean

# ── room OBJ loader ───────────────────────────────────────────────────────────

def _load_obj_cells(path):
    verts, objects = _parse_obj(path)
    if not verts or not objects:
        return []
    tv    = [Vertex.ByCoordinates(x, y, z) for (x, y, z) in verts]
    cells = []
    for obj_name, face_list, _ in objects:
        if not face_list:
            continue
        topo_faces = []
        for idx_list in face_list:
            n  = len(idx_list)
            if n < 3:
                continue
            fv = [tv[i] for i in idx_list]
            if n <= 4:
                f = _make_face(fv)
                if f:
                    topo_faces.append(f)
            else:
                f = _make_face(fv)
                if f:
                    topo_faces.append(f)
                else:
                    topo_faces.extend(_fan_tris(fv))
        c = _faces_to_cell(topo_faces)
        if c:
            cells.append(c)
    return cells

# ── main loading loop ─────────────────────────────────────────────────────────

selectors = []
all_cells = []

for fname, room_type in OBJ_TO_ROOM.items():
    obj_path = os.path.join(OBJECTS_DIR, fname + ".obj")
    if not os.path.exists(obj_path):
        print(f"  [SKIP] not found: {fname}.obj")
        continue

    cells_for_type = _load_obj_cells(obj_path)

    if not cells_for_type:
        print(f"  [SKIP] {fname}.obj - no cells extracted")
        continue

    label  = ROOM_LABEL[room_type]
    zoning = ZONING[room_type]
    conn   = NODE_CONNECTIVITY[room_type]
    color  = ROOM_COLOR.get(room_type, "#AAAAAA")

    for cell in cells_for_type:
        iv = Topology.InternalVertex(cell)
        d  = Dictionary.ByKeysValues(
            ["room_type", "label", "cell_color",
             "feat_zoning_type_0", "feat_zoning_type_1",
             "feat_zoning_type_2", "feat_zoning_type_3",
             "feat_connectivity_0", "feat_connectivity_1",
             "feat_connectivity_2"],
            [room_type, label, color,
             zoning[0], zoning[1], zoning[2], zoning[3],
             conn[0], conn[1], conn[2]]
        )
        iv = Topology.SetDictionary(iv, d)
        selectors.append(iv)
        all_cells.append(cell)

    print(f"  {fname:20s} -> {room_type:12s} label={label}  cells={len(cells_for_type)}")

print(f"Total: {len(all_cells)} cells, {len(selectors)} selectors")

Found 12 .obj file(s) in Objects folder
Recognized room files: 6
Skipped for room import (unmapped):
  - Entrance door.obj
  - Passage Door.obj
  - door.obj
  - door2.obj
  - rooms.obj
  - window.obj
Topology.Cells - Warning: The input is a Cell. Returning the same cell embedded in a list.
caller name: _faces_to_cell
Topology.Cells - Warning: The input is a Cell. Returning the same cell embedded in a list.
caller name: _faces_to_cell
Topology.Cells - Warning: The input is a Cell. Returning the same cell embedded in a list.
caller name: _faces_to_cell
Topology.Cells - Warning: The input is a Cell. Returning the same cell embedded in a list.
caller name: _faces_to_cell
Topology.Cells - Warning: The input is a Cell. Returning the same cell embedded in a list.
caller name: _faces_to_cell
Topology.Cells - Warning: The input is a Cell. Returning the same cell embedded in a list.
caller name: _faces_to_cell
  Bathroom.obj         -> bathroom     label=7 cells=6
Topology.Cells - Warning: The i

In [ ]:
# 6B. Robust room import fallback (handles difficult OBJ faces, e.g. Kitchen.obj)
def _polygon_faces_robust(tv, idx_list):
    n = len(idx_list)
    if n < 3:
        return []

    fv = [tv[i] for i in idx_list]
    f = _make_face(fv)
    if f:
        return [f]

    # If direct face creation fails, triangulate by fan as a geometric fallback.
    faces = []
    anchor = fv[0]
    for i in range(1, n - 1):
        tri = _make_face([anchor, fv[i], fv[i + 1]])
        if tri:
            faces.append(tri)
    return faces

def _faces_to_cell_robust(face_list):
    if not face_list:
        return None
    try:
        merged = Topology.SelfMerge(Cluster.ByTopologies(face_list))
        cells = Topology.Cells(merged)
        if cells:
            return cells[0]
        if Topology.IsInstance(merged, "Cell"):
            return merged
        if Topology.IsInstance(merged, "Shell"):
            c = Cell.ByShell(merged)
            if c:
                return c
        for sh in (Topology.Shells(merged) or []):
            c = Cell.ByShell(sh)
            if c:
                return c
    except Exception:
        pass

    try:
        # Last fallback path if merge/shell conversion fails.
        c = Cell.ByFaces(face_list)
        if c:
            return c
    except Exception:
        pass

    return None

def _proxy_cell_from_obj(path):
    """File-level fallback: create a bounding-box cell from OBJ vertices."""
    verts, _ = _parse_obj(path)
    if not verts:
        return None
    try:
        tv = [Vertex.ByCoordinates(x, y, z) for (x, y, z) in verts]
        bb = Topology.BoundingBox(Cluster.ByTopologies(tv))
        if Topology.IsInstance(bb, "Cell"):
            return bb
    except Exception:
        pass
    return None

def _proxy_cell_from_face_indices(tv, face_list):
    """Object-level fallback: bounding-box cell from vertices used by one OBJ object."""
    idx_set = {i for idx in face_list for i in idx if 0 <= i < len(tv)}
    if len(idx_set) < 4:
        return None
    try:
        verts = [tv[i] for i in sorted(idx_set)]
        bb = Topology.BoundingBox(Cluster.ByTopologies(verts))
        if Topology.IsInstance(bb, "Cell"):
            d = Dictionary.ByKeysValues(["_auto_proxy"], [1])
            bb = Topology.SetDictionary(bb, d)
            return bb
    except Exception:
        pass
    return None

def _load_obj_cells(path):
    verts, objects = _parse_obj(path)
    if not verts or not objects:
        return []

    tv = [Vertex.ByCoordinates(x, y, z) for (x, y, z) in verts]
    cells = []

    for obj_name, face_list, _ in objects:
        if not face_list:
            continue

        topo_faces = []
        for idx_list in face_list:
            topo_faces.extend(_polygon_faces_robust(tv, idx_list))

        c = _faces_to_cell_robust(topo_faces)
        if c:
            cells.append(c)
            continue

        # Recover failed object groups as proxy cells.
        proxy = _proxy_cell_from_face_indices(tv, face_list)
        if proxy:
            cells.append(proxy)

    return cells

def _centroid_vertex(top):
    try:
        c = Topology.Centroid(top)
        if c:
            return c
    except Exception:
        pass
    return Topology.InternalVertex(top)

def _dist_cells(a, b):
    va = _centroid_vertex(a)
    vb = _centroid_vertex(b)
    if va is None or vb is None:
        return 1e18
    ca = Vertex.Coordinates(va)
    cb = Vertex.Coordinates(vb)
    return ((ca[0]-cb[0])**2 + (ca[1]-cb[1])**2 + (ca[2]-cb[2])**2) ** 0.5

def _nearest_dist(cell, refs):
    if not refs:
        return 1e18
    return min(_dist_cells(cell, r) for r in refs)

# Rebuild selectors/all_cells using robust loader
selectors = []
all_cells = []
seed_cells_by_room = {}
seed_nonproxy_by_room = {}

all_obj_files = sorted([f for f in os.listdir(OBJECTS_DIR) if f.lower().endswith(".obj")])
recognized_room_files = []
skipped_obj_files = []

for obj_file in all_obj_files:
    stem = Path(obj_file).stem.strip().lower()
    room_type = OBJ_TO_ROOM.get(stem)
    if room_type is None:
        skipped_obj_files.append(obj_file)
        continue
    recognized_room_files.append((obj_file, room_type))

print(f"[Fallback pass] Recognized room files: {len(recognized_room_files)}")

for obj_file, room_type in recognized_room_files:
    obj_path = os.path.join(OBJECTS_DIR, obj_file)
    cells_for_type = _load_obj_cells(obj_path)

    # Last-resort file-level proxy if everything failed for this file.
    if not cells_for_type:
        proxy = _proxy_cell_from_obj(obj_path)
        if proxy:
            d0 = Dictionary.ByKeysValues(["_auto_proxy"], [1])
            proxy = Topology.SetDictionary(proxy, d0)
            cells_for_type = [proxy]
            print(f"  [PROXY FILE] {obj_file} - using bounding-box fallback cell")
        else:
            print(f"  [SKIP] {obj_file} - no cells extracted even with fallback")
            continue

    proxy_count = 0
    reassign_count = 0

    for cell in cells_for_type:
        d_cell = Topology.Dictionary(cell)
        is_proxy = 0
        if d_cell is not None:
            try:
                is_proxy = int(Dictionary.ValueAtKey(d_cell, "_auto_proxy") or 0)
            except Exception:
                is_proxy = 0

        effective_room_type = room_type

        if is_proxy and room_type == "corridor":
            # Heuristic: uncertain corridor proxies near bedroom seeds are bedrooms, not corridors.
            d_bed = _nearest_dist(cell, seed_nonproxy_by_room.get("bedroom", []))
            d_cor = _nearest_dist(cell, seed_nonproxy_by_room.get("corridor", []))
            if d_bed < 0.70 * d_cor:
                effective_room_type = "bedroom"
                reassign_count += 1

        label = ROOM_LABEL[effective_room_type]
        zoning = ZONING[effective_room_type]
        conn = NODE_CONNECTIVITY[effective_room_type]
        color = ROOM_COLOR.get(effective_room_type, "#AAAAAA")

        d = Dictionary.ByKeysValues(
            [
                "room_type",
                "label",
                "cell_color",
                "feat_zoning_type_0",
                "feat_zoning_type_1",
                "feat_zoning_type_2",
                "feat_zoning_type_3",
                "feat_connectivity_0",
                "feat_connectivity_1",
                "feat_connectivity_2",
            ],
            [
                effective_room_type,
                label,
                color,
                zoning[0],
                zoning[1],
                zoning[2],
                zoning[3],
                conn[0],
                conn[1],
                conn[2],
            ],
        )

        if is_proxy:
            proxy_count += 1
            d = Dictionary.SetValuesAtKeys(
                d,
                ["is_proxy_room", "proxy_room_type"],
                [1, effective_room_type],
            )
            cell = Topology.SetDictionary(cell, d)
            all_cells.append(cell)
            seed_cells_by_room.setdefault(effective_room_type, []).append(cell)
        else:
            iv = Topology.InternalVertex(cell)
            iv = Topology.SetDictionary(iv, d)
            selectors.append(iv)
            all_cells.append(cell)
            seed_cells_by_room.setdefault(effective_room_type, []).append(cell)
            seed_nonproxy_by_room.setdefault(effective_room_type, []).append(cell)

    loaded_source = sum(1 for c in seed_cells_by_room.get(room_type, []))
    if proxy_count > 0 or reassign_count > 0:
        print(
            f"  {obj_file:20s} -> {room_type:12s} loaded={loaded_source} "
            f"(proxy={proxy_count}, reassigned={reassign_count})"
        )
    else:
        print(f"  {obj_file:20s} -> {room_type:12s} loaded={loaded_source}")

print(f"\n[Fallback pass] Total: {len(all_cells)} cells, {len(selectors)} selectors")

[Fallback pass] Recognized room files: 6
Topology.Cells - Warning: The input is a Cell. Returning the same cell embedded in a list.
caller name: _faces_to_cell_robust
Topology.Cells - Warning: The input is a Cell. Returning the same cell embedded in a list.
caller name: _faces_to_cell_robust
Topology.Cells - Warning: The input is a Cell. Returning the same cell embedded in a list.
caller name: _faces_to_cell_robust
Topology.Cells - Warning: The input is a Cell. Returning the same cell embedded in a list.
caller name: _faces_to_cell_robust
Topology.Cells - Warning: The input is a Cell. Returning the same cell embedded in a list.
caller name: _faces_to_cell_robust
Topology.Cells - Warning: The input is a Cell. Returning the same cell embedded in a list.
caller name: _faces_to_cell_robust
  Bathroom.obj         -> bathroom     loaded=6
Topology.Cells - Warning: The input is a Cell. Returning the same cell embedded in a list.
caller name: _faces_to_cell_robust
Topology.Cells - Warning: The

## 7. Build CellComplex and transfer room-type dictionaries

All room cells are merged into a single `CellComplex`. Then
`Topology.TransferDictionariesBySelectors` assigns the room-type dictionaries from the
selector vertices to the CellComplex cells.

In [180]:
cc = Topology.SelfMerge(Cluster.ByTopologies(all_cells))
print("Merged topology type:", Topology.TypeAsString(cc))

cc_cells = Topology.Cells(cc)
print(f"CellComplex cells : {len(cc_cells)}")
print(f"CellComplex faces : {len(Topology.Faces(cc))}")

# Use a tighter tolerance to reduce accidental cross-room dictionary mixing.
cc = Topology.TransferDictionariesBySelectors(cc, selectors, tranCells=True, tolerance=0.001)

def _scalar(value):
    if isinstance(value, list):
        return value[0] if value else None
    return value

# Normalize cell dictionaries so downstream code gets scalar values, not lists.
fixed_cells = []
for cell in Topology.Cells(cc):
    d = Topology.Dictionary(cell)

    is_proxy = _scalar(Dictionary.ValueAtKey(d, "is_proxy_room"))
    proxy_room_type = _scalar(Dictionary.ValueAtKey(d, "proxy_room_type"))

    room_type = _scalar(Dictionary.ValueAtKey(d, "room_type"))
    label = _scalar(Dictionary.ValueAtKey(d, "label"))
    cell_color = _scalar(Dictionary.ValueAtKey(d, "cell_color"))

    z0 = _scalar(Dictionary.ValueAtKey(d, "feat_zoning_type_0"))
    z1 = _scalar(Dictionary.ValueAtKey(d, "feat_zoning_type_1"))
    z2 = _scalar(Dictionary.ValueAtKey(d, "feat_zoning_type_2"))
    z3 = _scalar(Dictionary.ValueAtKey(d, "feat_zoning_type_3"))
    c0 = _scalar(Dictionary.ValueAtKey(d, "feat_connectivity_0"))
    c1 = _scalar(Dictionary.ValueAtKey(d, "feat_connectivity_1"))
    c2 = _scalar(Dictionary.ValueAtKey(d, "feat_connectivity_2"))

    if is_proxy and proxy_room_type:
        # Re-assert proxy room metadata in case selector transfer overwrote it.
        room_type = proxy_room_type
        label = ROOM_LABEL.get(room_type, 0)
        cell_color = ROOM_COLOR.get(room_type, "#AAAAAA")
        z = ZONING.get(room_type, [0, 0, 0, 0])
        c = NODE_CONNECTIVITY.get(room_type, [0, 1, 0])
        z0, z1, z2, z3 = z
        c0, c1, c2 = c

    d = Dictionary.SetValuesAtKeys(
        d,
        [
            "room_type",
            "label",
            "cell_color",
            "feat_zoning_type_0",
            "feat_zoning_type_1",
            "feat_zoning_type_2",
            "feat_zoning_type_3",
            "feat_connectivity_0",
            "feat_connectivity_1",
            "feat_connectivity_2",
        ],
        [room_type, label, cell_color, z0, z1, z2, z3, c0, c1, c2],
    )
    cell = Topology.SetDictionary(cell, d)
    fixed_cells.append(cell)

cc = Topology.SelfMerge(Cluster.ByTopologies(fixed_cells))

# Diagnostic: count how many cells got each room_type
from collections import Counter
label_counts = Counter()
unlabelled = 0
for cell in Topology.Cells(cc):
    d = Topology.Dictionary(cell)
    rt = _scalar(Dictionary.ValueAtKey(d, "room_type"))
    if rt:
        label_counts[rt] += 1
    else:
        unlabelled += 1

print("\nRoom-type distribution after dictionary transfer:")
LABEL_NAME = {v: k for k, v in ROOM_LABEL.items()}
for rt, cnt in sorted(label_counts.items(), key=lambda x: ROOM_LABEL.get(x[0], 99)):
    print(f"  {rt:15s} (label {ROOM_LABEL[rt]}) - {cnt} cell(s)")
if unlabelled:
    print(f"  *** {unlabelled} cell(s) without room_type (dictionary transfer missed them)")
else:
    print("  All cells labelled correctly.")

Merged topology type: Cluster
CellComplex cells : 23
CellComplex faces : 148

Room-type distribution after dictionary transfer:
  bedroom         (label 0) - 7 cell(s)
  livingroom      (label 1) - 2 cell(s)
  kitchen         (label 2) - 1 cell(s)
  corridor        (label 4) - 3 cell(s)
  stairs          (label 5) - 4 cell(s)
  bathroom        (label 7) - 6 cell(s)
  All cells labelled correctly.


## 8. Visualise the CellComplex (coloured by room type)

In [168]:
cc_cells = Topology.Cells(cc)

# Build a flat list of clean display faces — merge_coplanar_for_display() converts
# each cell's triangulated faces into clean planar quads/polygons WITHOUT rebuilding
# the Cell topology (so cells used for graph construction are unchanged).
display_faces = []
for cell in cc_cells:
    d_cell    = Topology.Dictionary(cell)
    color     = Dictionary.ValueAtKey(d_cell, "cell_color") or "#AAAAAA"
    raw_faces = Topology.Faces(cell) or []
    for f in merge_coplanar_for_display(raw_faces):
        f = Topology.SetDictionary(f, Dictionary.ByKeysValues(["cell_color"], [color]))
        display_faces.append(f)

Topology.Show(
    display_faces,
    faceColorKey="cell_color",
    faceOpacity=0.6,
    backgroundColor="white",
    width=900,
    height=700,
    renderer=renderer
)

## 9. Load door OBJs as apertures

Doors are modelled as planar face objects. We collect the faces from `door.obj` and
`Entrance door.obj`, tag each face with a `door_type` dictionary, then pass them as
apertures to `Topology.AddApertures`.

The apertures are placed on whichever CellComplex face they coincide with (within tolerance).
When `Graph.ByTopology(cc, directApertures=True)` is later called, it creates graph edges
only between cells whose shared face carries an aperture — modelling room connectivity
through actual door openings.

In [169]:
DOOR_OBJS = {
    "door":    os.path.join(OBJECTS_DIR, "door2.obj"),
    "passage": os.path.join(OBJECTS_DIR, "Passage Door.obj"),
}

apertures = []

for door_type, obj_path in DOOR_OBJS.items():
    if not os.path.exists(obj_path):
        print(f"  [SKIP] not found: {obj_path}")
        continue

    conn  = DOOR_CONNECTIVITY[door_type]
    verts, objects = _parse_obj(obj_path)
    if not verts or not objects:
        print(f"  [SKIP] {door_type}: nothing parsed")
        continue

    tv = [Vertex.ByCoordinates(x, y, z) for (x, y, z) in verts]
    faces_for_type = []

    for obj_name, face_list, line_list in objects:
        for idx_list in face_list:
            n = len(idx_list)
            if n < 3:
                continue
            fv = [tv[i] for i in idx_list]
            f  = _make_face(fv)
            if f:
                faces_for_type.append(f)

        for idx_list in line_list:
            unique = list(idx_list)
            if len(unique) >= 2 and unique[0] == unique[-1]:
                unique = unique[:-1]
            if len(unique) < 3:
                continue
            fv = [tv[i] for i in unique]
            f  = _make_face(fv)
            if f:
                faces_for_type.append(f)

    for f in faces_for_type:
        d = Dictionary.ByKeysValues(
            ["door_type",
             "feat_connectivity_0", "feat_connectivity_1", "feat_connectivity_2"],
            [door_type, conn[0], conn[1], conn[2]]
        )
        f = Topology.SetDictionary(f, d)
        apertures.append(f)

    print(f"  {door_type:20s} -> {len(faces_for_type)} aperture faces")

print(f"Total apertures: {len(apertures)}")


Door OBJ files detected: 4
  - Entrance door.obj -> entrance_door
  - Passage Door.obj -> passage
  - door.obj -> door
  - door2.obj -> door
  Entrance door.obj    -> entrance_door aperture faces=1
  Passage Door.obj     -> passage       aperture faces=9
  door.obj             -> door          aperture faces=20
  door2.obj            -> door          aperture faces=11

Total apertures: 41


## 10. Add apertures to CellComplex

In [170]:
if apertures:
    cc = Topology.AddApertures(
        cc,
        apertures,
        exclusive=False,
        subTopologyType="Face",
        tolerance=0.001
    )
    print("Apertures added to CellComplex.")
else:
    print("No apertures to add.")

# Count how many faces now carry apertures
faces_with_apertures = []
for f in (Topology.Faces(cc) or []):
    aps = Topology.Apertures(f)
    if aps:
        faces_with_apertures.append(f)
print(f"Faces carrying apertures: {len(faces_with_apertures)}")

Apertures added to CellComplex.
Faces carrying apertures: 20


## 11. Build the room adjacency graph

`Graph.ByTopology` with `directApertures=True` creates:
- One **vertex** per cell (room)  
- One **edge** between two cells whose shared face carries a door aperture

This exactly mirrors the MSD dataset construction.

In [171]:
graph = Graph.ByTopology(
    cc,
    direct=False,
    directApertures=True,
    viaSharedTopologies=False,
    viaSharedApertures=False,
    toExteriorTopologies=False,
    toExteriorApertures=False,
    tolerance=0.001
)

n_verts = len(Graph.Vertices(graph) or [])
n_edges = len(Graph.Edges(graph) or [])
print(f"Graph: {n_verts} vertices (rooms), {n_edges} edges (door connections)")

# Fallback: if no edges found (geometry mismatch), use direct adjacency
if n_edges == 0:
    print("\nNo aperture-based edges found — falling back to direct cell adjacency.")
    graph = Graph.ByTopology(
        cc,
        direct=True,
        directApertures=False,
        tolerance=0.001
    )
    n_verts = len(Graph.Vertices(graph) or [])
    n_edges = len(Graph.Edges(graph) or [])
    print(f"Fallback graph: {n_verts} vertices, {n_edges} edges")

Graph: 23 vertices (rooms), 0 edges (door connections)

No aperture-based edges found — falling back to direct cell adjacency.
Fallback graph: 23 vertices, 0 edges


## 12. Verify graph — check vertex dictionaries

In [172]:
vertices = Graph.Vertices(graph)
edges    = Graph.Edges(graph)

print("Sample vertex dictionaries:")
for v in vertices[:6]:
    d = Topology.Dictionary(v)
    rt    = Dictionary.ValueAtKey(d, "room_type")
    label = Dictionary.ValueAtKey(d, "label")
    z0    = Dictionary.ValueAtKey(d, "feat_zoning_type_0")
    c0    = Dictionary.ValueAtKey(d, "feat_connectivity_0")
    print(f"  room_type={rt}, label={label}, z0={z0}, c0={c0}")

print("\nSample edge dictionaries:")
for e in edges[:4]:
    d = Topology.Dictionary(e)
    print(f"  keys: {Dictionary.Keys(d)}")

Sample vertex dictionaries:
  room_type=bathroom, label=7, z0=0, c0=0
  room_type=bathroom, label=7, z0=0, c0=0
  room_type=bathroom, label=7, z0=0, c0=0
  room_type=bathroom, label=7, z0=0, c0=0
  room_type=bathroom, label=7, z0=0, c0=0
  room_type=bathroom, label=7, z0=0, c0=0

Sample edge dictionaries:


## 13. Propagate aperture connectivity features to graph edges

When `Graph.ByTopology(directApertures=True)` creates an edge, the edge is placed at the
centroid of the shared aperture face. We find the nearest aperture to each edge and copy its
`feat_connectivity_*` values. Edges from the fallback (no apertures) default to `door`.

In [173]:
import math

def _dist(v1, v2):
    c1, c2 = Vertex.Coordinates(v1), Vertex.Coordinates(v2)
    return math.sqrt(sum((a - b) ** 2 for a, b in zip(c1, c2)))

def _edge_centroid(e):
    sv = Edge.StartVertex(e)
    ev = Edge.EndVertex(e)
    sc, ec = Vertex.Coordinates(sv), Vertex.Coordinates(ev)
    return Vertex.ByCoordinates(
        (sc[0] + ec[0]) / 2,
        (sc[1] + ec[1]) / 2,
        (sc[2] + ec[2]) / 2
    )

def _face_centroid(f):
    vs = Topology.Vertices(f)
    coords = [Vertex.Coordinates(v) for v in vs]
    n = len(coords)
    return Vertex.ByCoordinates(
        sum(c[0] for c in coords) / n,
        sum(c[1] for c in coords) / n,
        sum(c[2] for c in coords) / n
    )

# Pre-compute aperture centroids
aperture_centroids = []
for ap in apertures:
    c = _face_centroid(ap)
    d = Topology.Dictionary(ap)
    aperture_centroids.append((c, d))

# Assign connectivity to each edge
updated_edges = []
default_conn = DOOR_CONNECTIVITY["door"]  # fallback

for e in edges:
    ec = _edge_centroid(e)
    d  = Topology.Dictionary(e)

    # Check if connectivity already set
    existing = Dictionary.ValueAtKey(d, "feat_connectivity_0")
    if existing is not None:
        updated_edges.append(e)
        continue

    # Find nearest aperture
    if aperture_centroids:
        best_d, best_ap_dict = min(
            ((_dist(ec, ac), ad) for ac, ad in aperture_centroids),
            key=lambda x: x[0]
        )
        conn_keys = ["feat_connectivity_0", "feat_connectivity_1", "feat_connectivity_2"]
        conn_vals = [
            Dictionary.ValueAtKey(best_ap_dict, k) or default_conn[i]
            for i, k in enumerate(conn_keys)
        ]
        door_type = Dictionary.ValueAtKey(best_ap_dict, "door_type") or "door"
    else:
        conn_vals  = default_conn
        door_type  = "door"

    d = Dictionary.SetValuesAtKeys(
        d,
        ["door_type",
         "feat_connectivity_0", "feat_connectivity_1", "feat_connectivity_2"],
        [door_type, conn_vals[0], conn_vals[1], conn_vals[2]]
    )
    e = Topology.SetDictionary(e, d)
    updated_edges.append(e)

print(f"{len(updated_edges)} edges with connectivity features assigned.")

0 edges with connectivity features assigned.


## 14. Visualise the room adjacency graph

In [174]:
# Apply colors to vertices for display
for v in vertices:
    d  = Topology.Dictionary(v)
    rt = Dictionary.ValueAtKey(d, "room_type") or "unknown"
    d  = Dictionary.SetValuesAtKeys(d,
        ["color", "size"], [ROOM_COLOR.get(rt, "#AAAAAA"), 12])
    v  = Topology.SetDictionary(v, d)

Topology.Show(
    graph,
    cc_cells,
    vertexColorKey="color",
    vertexSize=8,
    faceColorKey="cell_color",
    faceOpacity=0.2,
    backgroundColor="white",
    width=900,
    height=700,
    renderer=renderer
)

## 15. Export CSVs in MSD model schema

The pretrained `msd_node_classifier.pt` expects exactly:

| File | Columns |
|---|---|
| `graphs.csv` | `graph_id`, `num_nodes` |
| `nodes.csv` | `graph_id`, `node_id`, `label`, `feat_zoning_type_0..3`, `feat_connectivity_0..2`, `train_mask`, `val_mask`, `test_mask` |
| `edges.csv` | `graph_id`, `src_id`, `dst_id`, `feat_connectivity_0..2` |

In [175]:
# Build a vertex → index lookup using rounded coordinates
def _vkey(v, tol=3):
    c = Vertex.Coordinates(v)
    return tuple(round(x, tol) for x in c)

vert_idx = {_vkey(v): i for i, v in enumerate(vertices)}

# ---------- nodes.csv ----------
nodes_rows = []
for i, v in enumerate(vertices):
    d = Topology.Dictionary(v)
    label = Dictionary.ValueAtKey(d, "label")
    if label is None:
        label = 0  # unknown room defaults to bedroom index

    def gv(key, default=0):
        val = Dictionary.ValueAtKey(d, key)
        return val if val is not None else default

    nodes_rows.append({
        "graph_id":            0,
        "node_id":             i,
        "label":               int(label),
        "feat_zoning_type_0":  int(gv("feat_zoning_type_0")),
        "feat_zoning_type_1":  int(gv("feat_zoning_type_1")),
        "feat_zoning_type_2":  int(gv("feat_zoning_type_2")),
        "feat_zoning_type_3":  int(gv("feat_zoning_type_3")),
        "feat_connectivity_0": int(gv("feat_connectivity_0")),
        "feat_connectivity_1": int(gv("feat_connectivity_1", 1)),  # default: door
        "feat_connectivity_2": int(gv("feat_connectivity_2")),
        "train_mask":          0,
        "val_mask":            0,
        "test_mask":           1,
    })

# ---------- edges.csv ----------
edges_rows = []
for e in updated_edges:
    sv_key = _vkey(Edge.StartVertex(e))
    ev_key = _vkey(Edge.EndVertex(e))
    src_id = vert_idx.get(sv_key)
    dst_id = vert_idx.get(ev_key)
    if src_id is None or dst_id is None:
        continue

    d = Topology.Dictionary(e)

    def ge(key, default=0):
        val = Dictionary.ValueAtKey(d, key)
        return val if val is not None else default

    edges_rows.append({
        "graph_id":            0,
        "src_id":              src_id,
        "dst_id":              dst_id,
        "feat_connectivity_0": int(ge("feat_connectivity_0")),
        "feat_connectivity_1": int(ge("feat_connectivity_1", 1)),  # default: door
        "feat_connectivity_2": int(ge("feat_connectivity_2")),
    })

# ---------- graphs.csv ----------
graphs_rows = [{"graph_id": 0, "num_nodes": len(vertices)}]

# Write files
pd.DataFrame(graphs_rows).to_csv(os.path.join(DATASET_PATH, "graphs.csv"), index=False)
pd.DataFrame(nodes_rows ).to_csv(os.path.join(DATASET_PATH, "nodes.csv" ), index=False)
pd.DataFrame(edges_rows ).to_csv(os.path.join(DATASET_PATH, "edges.csv" ), index=False)

print(f"graphs.csv : 1 graph")
print(f"nodes.csv  : {len(nodes_rows)} nodes")
print(f"edges.csv  : {len(edges_rows)} edges")
print(f"Saved to   : {DATASET_PATH}")

graphs.csv : 1 graph
nodes.csv  : 23 nodes
edges.csv  : 0 edges
Saved to   : C:\Users\etmaglari\IAAC\etmaglari_gML\Homework04\dataset_node_classification


## 16. Inspect the exported CSV schema

In [176]:
nodes_df  = pd.read_csv(os.path.join(DATASET_PATH, "nodes.csv"))
edges_df  = pd.read_csv(os.path.join(DATASET_PATH, "edges.csv"))
graphs_df = pd.read_csv(os.path.join(DATASET_PATH, "graphs.csv"))

print("graphs.csv columns:", list(graphs_df.columns))
print(graphs_df.to_string(index=False))
print()
print("nodes.csv columns:", list(nodes_df.columns))
print(nodes_df.to_string(index=False))
print()
print("edges.csv columns:", list(edges_df.columns))
print(edges_df.head(10).to_string(index=False))

EmptyDataError: No columns to parse from file

## 17. Load dataset into PyG

`PyG.ByCSVPath` reads the three CSVs and builds a PyTorch Geometric `Data` object.

In [ ]:
pyg = PyG.ByCSVPath(
    path=DATASET_PATH,
    level="node",
    task="classification",
    graphLabelType="categorical",
    nodeLabelType="categorical",
    edgeLabelType="categorical"
)
print("Dataset loaded:", pyg)

Dataset loaded: <topologicpy.PyG.PyG object at 0x000001E3DD00C150>


## 18. Load the pretrained MSD node classifier

In [ ]:
pyg.LoadModel(MODEL_PATH)
print("Model loaded from:", MODEL_PATH)

Model loaded from: C:\Users\etmaglari\IAAC\etmaglari_gML\S0 Classes\msd-main\msd_node_classifier.pt


## 19. Predict room types

In [ ]:
pred_report = pyg.Predict(split="all", return_probs=True, attach_to_data=True)
print("Predictions complete.")

Predictions complete.


## 20. Export node-level predictions to CSV

In [ ]:
import numpy as np

def _to_class_index(value):
    arr = np.asarray(value)
    arr = np.squeeze(arr)
    if arr.ndim == 0:
        return int(arr)
    if arr.ndim == 1:
        if arr.size == 1:
            return int(arr[0])
        return int(np.argmax(arr))
    raise ValueError(f"Cannot convert shape {arr.shape} to class index.")

def export_node_predictions(pyg_obj, output_csv):
    report       = pyg_obj.Predict(split="all", return_probs=True, attach_to_data=True)
    pred_list    = report["pred"]
    true_list    = report["y_true"]
    prob_list    = report.get("prob", None)
    rows = []
    for g_idx, data in enumerate(pyg_obj.data_list):
        graph_id = int(data.graph_id.item()) if hasattr(data, "graph_id") else g_idx
        n        = data.num_nodes
        g_pred   = np.asarray(pred_list[g_idx])
        g_true   = np.asarray(true_list[g_idx])
        g_prob   = np.asarray(prob_list[g_idx]) if prob_list is not None else None
        for ni in range(n):
            y_true = _to_class_index(g_true[ni])
            y_pred = _to_class_index(g_pred[ni])
            row    = {"graph_id": graph_id, "node_id": ni,
                      "y_true": y_true, "y_pred": y_pred}
            if g_prob is not None:
                p = np.asarray(g_prob[ni]).squeeze()
                if p.ndim == 1 and y_pred < p.size:
                    row["y_pred_prob"] = float(p[y_pred])
            rows.append(row)
    df = pd.DataFrame(rows)
    df.to_csv(output_csv, index=False)
    return df

predictions_df = export_node_predictions(pyg, PRED_CSV)
print(f"Predictions saved to: {PRED_CSV}")

# Label index → name for display
LABEL_NAME = {v: k for k, v in ROOM_LABEL.items()}
predictions_df["true_name"] = predictions_df["y_true"].map(LABEL_NAME)
predictions_df["pred_name"] = predictions_df["y_pred"].map(LABEL_NAME)
correct = (predictions_df["y_true"] == predictions_df["y_pred"]).sum()
total   = len(predictions_df)
print(f"\nAccuracy: {correct}/{total} = {correct/total:.1%}")
print(predictions_df[["node_id", "true_name", "pred_name"]].to_string(index=False))

Predictions saved to: C:\Users\etmaglari\IAAC\etmaglari_gML\Homework04\dataset_node_classification\node_predictions_homework04.csv

Accuracy: 11/28 = 39.3%
 node_id  true_name pred_name
       0    bedroom   bedroom
       1   bathroom  bathroom
       2    bedroom    stairs
       3   bathroom    stairs
       4   bathroom    stairs
       5    bedroom   bedroom
       6    bedroom   bedroom
       7    bedroom    stairs
       8   bathroom    stairs
       9    bedroom    stairs
      10    bedroom    stairs
      11 livingroom    dining
      12    bedroom    stairs
      13    bedroom   bedroom
      14   corridor  corridor
      15     stairs storeroom
      16   bathroom    stairs
      17    bedroom    stairs
      18 livingroom    dining
      19    bedroom    stairs
      20    bedroom    stairs
      21   corridor  corridor
      22     stairs storeroom
      23     stairs    stairs
      24   corridor   kitchen
      25   bathroom  bathroom
      26    bedroom   bedroom
    

## 21. Prepare dataset for visualisation

We merge the predictions back into the nodes.csv — renaming `label` → `true` and adding
a `pred` column — then save to a `dataset_for_visualisation/` folder.
`Graph.ByCSVPath` reads this and populates `true` / `pred` on each vertex dictionary.

In [ ]:
VIS_PATH = os.path.join(os.path.dirname(DATASET_PATH), "dataset_for_visualisation")
os.makedirs(VIS_PATH, exist_ok=True)

# Merge predictions into nodes_df
nodes_vis = nodes_df.copy()
nodes_vis = nodes_vis.rename(columns={"label": "true"})
nodes_vis["pred"] = predictions_df["y_pred"].values

nodes_vis.to_csv(os.path.join(VIS_PATH, "nodes.csv" ), index=False)
graphs_df .to_csv(os.path.join(VIS_PATH, "graphs.csv"), index=False)
edges_df  .to_csv(os.path.join(VIS_PATH, "edges.csv" ), index=False)

print(f"Visualisation dataset saved to: {VIS_PATH}")
print("nodes.csv columns:", list(nodes_vis.columns))

Visualisation dataset saved to: C:\Users\etmaglari\IAAC\etmaglari_gML\Homework04\dataset_for_visualisation
nodes.csv columns: ['graph_id', 'node_id', 'true', 'feat_zoning_type_0', 'feat_zoning_type_1', 'feat_zoning_type_2', 'feat_zoning_type_3', 'feat_connectivity_0', 'feat_connectivity_1', 'feat_connectivity_2', 'train_mask', 'val_mask', 'test_mask', 'pred']


## 22. Visualise true vs predicted labels

Vertices coloured by **true** room type (left view) and **predicted** room type (right view).
Misclassified nodes are shown in **red** at size 30; correct nodes use a thermal colour scale.

In [ ]:
vis_graphs = Graph.ByCSVPath(path=VIS_PATH)
print(f"{len(vis_graphs)} graph(s) loaded for visualisation.")

g = vis_graphs[0]
verts = Graph.Vertices(g)

for v in verts:
    d    = Topology.Dictionary(v)
    true = int(Dictionary.ValueAtKey(d, "true") or 0)
    pred = int(Dictionary.ValueAtKey(d, "pred") or 0)
    if true != pred:
        size        = 30
        true_color  = "red"
        pred_color  = "red"
    else:
        size        = 14
        true_color  = Color.ByValueInRange(true, minValue=0, maxValue=8)
        pred_color  = Color.ByValueInRange(pred, minValue=0, maxValue=8)
    d = Dictionary.SetValuesAtKeys(
        d,
        ["true_color", "pred_color", "size", "true", "pred"],
        [true_color, pred_color, size, true, pred]
    )
    v = Topology.SetDictionary(v, d)

g = Graph.Reshape(g)

print("--- True labels ---")
Topology.Show(
    g,
    vertexSize=6,
    vertexSizeKey="size",
    vertexColorKey="true_color",
    showVertexLabel=True,
    vertexLabelKey="true",
    vertexLabelFontSize=18,
    backgroundColor="white",
    camera=[0, 0, 3],
    width=900,
    height=600,
    renderer=renderer
)

print("--- Predicted labels ---")
Topology.Show(
    g,
    vertexSize=6,
    vertexSizeKey="size",
    vertexColorKey="pred_color",
    showVertexLabel=True,
    vertexLabelKey="pred",
    vertexLabelFontSize=18,
    backgroundColor="white",
    camera=[0, 0, 3],
    width=900,
    height=600,
    renderer=renderer
)

1 graph(s) loaded for visualisation.
--- True labels ---


--- Predicted labels ---
